# Bài 12: Đạo hàm và Phát hiện biên (Differentiation and Edge Detection)

Nói một cách không chính thức, biên (edge) là nơi cường độ thay đổi nhanh. Đó là một phát biểu về *đạo hàm*. Bài này xây dựng các phép đạo hàm ảnh rời rạc &mdash; các toán tử Prewitt, Sobel và Scharr &mdash; dưới dạng kernel tích chập (Bài 10), dùng chúng bên trong bộ phát hiện biên **Canny** kinh điển, và kết thúc bằng việc biến các bản đồ biên thu được thành các hình dạng thực sự nhờ **biến đổi Hough (Hough transform)**.

In [ ]:
# Download the sample images used in this lesson, if not already present
import os, urllib.request

for _f in ['calvin.png']:
    _path = f"../img/{_f}"
    if not os.path.exists(_path):
        os.makedirs("../img", exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/sbirchfield/sbirchfield.github.io/main/cvintro/img/{_f}", _path)

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Từ đạo hàm đến gradient

Hãy coi ảnh là một hàm số $I(x,y)$. Gradient của nó $\nabla I = (I_x, I_y)$ hướng theo chiều cường độ tăng nhanh nhất, trong đó $I_x = \partial I/\partial x$ và $I_y = \partial I/\partial y$ là các **đạo hàm riêng (partial derivatives)** của cường độ ảnh theo phương ngang và phương dọc &mdash; cường độ thay đổi nhanh đến mức nào khi bạn di chuyển sang phải một điểm ảnh, hoặc xuống dưới một điểm ảnh. Hai con số tóm tắt gradient tại mỗi điểm ảnh:

- **độ lớn (magnitude)** $\|\nabla I\| = \sqrt{I_x^2 + I_y^2}$ &mdash; mức độ thay đổi gắt đến đâu (lớn ở biên, gần 0 ở vùng phẳng)
- **hướng (direction)** $\theta = \mathrm{atan2}(I_y, I_x)$ &mdash; cường độ đang tăng nhanh nhất theo hướng nào (vuông góc với biên)

Bản thân $I_x$ và $I_y$ được tính bằng cách tích chập ảnh với các kernel đạo hàm nhỏ.

In [ ]:
def make_test_image(size=200):
    img = np.zeros((size, size), dtype=np.uint8)
    cv2.rectangle(img, (30, 30), (100, 100), 200, -1)
    cv2.circle(img, (140, 140), 40, 150, -1)
    pts = np.array([[150, 30], [190, 90], [110, 90]], dtype=np.int32)
    cv2.fillPoly(img, [pts], 25)  # low-contrast triangle, for the threshold demo later
    return img

img = make_test_image()
plt.imshow(img, cmap='gray')
plt.title('Test image (edges at several orientations)')
plt.axis('off')
plt.show()

## Đạo hàm ngây thơ: chỉ là một hiệu số

Ước lượng $I_x$ đơn giản nhất là kernel sai phân trung tâm 1D $[-1,\ 0,\ 1]$. Nó hoạt động được, nhưng vì không làm trơn theo phương vuông góc, nó nhạy với nhiễu ở mức tối đa.

In [ ]:
simple_x = np.array([[-1, 0, 1]], dtype=np.float64)
simple_y = simple_x.T

gx = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_x)
gy = cv2.filter2D(img.astype(np.float64), cv2.CV_64F, simple_y)
magnitude = np.sqrt(gx**2 + gy**2)

plt.imshow(magnitude, cmap='gray')
plt.title('Gradient magnitude, simple [-1, 0, 1] kernel')
plt.axis('off')
plt.show()

## Prewitt, Sobel và Scharr: đạo hàm + làm trơn

Các toán tử thực tế kết hợp phép lấy hiệu theo một phương với phép *làm trơn* lấy trung bình theo phương vuông góc &mdash; đây là điều giúp chúng chống chịu tốt với nhiễu. Chúng chỉ khác nhau ở cách đánh trọng số cho phép làm trơn đó:

| Toán tử | Kernel theo $x$ | Trọng số theo phương vuông góc |
|---|---|---|
| **Prewitt** | $\begin{bmatrix}-1&0&1\\-1&0&1\\-1&0&1\end{bmatrix}$ | đều: 1, 1, 1 |
| **Sobel** | $\begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix}$ | nhị thức: 1, 2, 1 |
| **Scharr** | $\begin{bmatrix}-3&0&3\\-10&0&10\\-3&0&3\end{bmatrix}$ | 3, 10, 3 |

Mỗi kernel theo $y$ chỉ là chuyển vị của kernel theo $x$ tương ứng. Các trọng số nhị thức của Sobel xấp xỉ một hàm Gauss nhỏ, đó là lý do nó là lựa chọn mặc định phổ biến nhất. Các trọng số của Scharr được chọn (tối ưu hóa bằng số) đặc biệt để làm cho *hướng gradient ước lượng được* chính xác nhất có thể theo mọi góc xoay &mdash; tức là gần với đạo hàm liên tục thực sự nhất mà một kernel nguyên $3\times3$ có thể đạt được &mdash; điều này quan trọng nhất trong các ứng dụng như luồng quang học (optical flow), vốn phụ thuộc vào góc gradient chính xác chứ không chỉ vào vị trí biên.

In [ ]:
prewitt_x = np.array([[-1, 0, 1]] * 3, dtype=np.float64)
prewitt_y = prewitt_x.T

operators = {
    'Simple': (simple_x, simple_y),
    'Prewitt': (prewitt_x, prewitt_y),
    'Sobel': None,   # use cv2.Sobel directly below
    'Scharr': None,  # use cv2.Scharr directly below
}

img_f = img.astype(np.float64)
magnitudes = {}
magnitudes['Simple'] = magnitude
magnitudes['Prewitt'] = np.sqrt(cv2.filter2D(img_f, cv2.CV_64F, prewitt_x)**2 +
                                 cv2.filter2D(img_f, cv2.CV_64F, prewitt_y)**2)
magnitudes['Sobel'] = np.sqrt(cv2.Sobel(img_f, cv2.CV_64F, 1, 0, ksize=3)**2 +
                               cv2.Sobel(img_f, cv2.CV_64F, 0, 1, ksize=3)**2)
magnitudes['Scharr'] = np.sqrt(cv2.Scharr(img_f, cv2.CV_64F, 1, 0)**2 +
                                cv2.Scharr(img_f, cv2.CV_64F, 0, 1)**2)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.5))
for ax, (name, mag) in zip(axes, magnitudes.items()):
    ax.imshow(mag, cmap='gray')
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

Trên một ảnh sạch, ít nhiễu như thế này, bốn kết quả trông gần như giống hệt nhau &mdash; khác biệt thực tế giữa chúng rất tinh tế và chủ yếu thể hiện khi có nhiễu hoặc khi cần góc chính xác đến mức dưới điểm ảnh (sub-pixel), chứ không phải là một bản đồ biên khác biệt thấy rõ.

### Khả năng chống nhiễu: tại sao việc làm trơn theo phương vuông góc lại quan trọng

Để thấy cụ thể lợi ích của việc làm trơn theo phương vuông góc, ta đo phản hồi của mỗi kernel với nhiễu thuần túy trên một vùng phẳng (không có biên). Để so sánh công bằng, trước tiên ta chuẩn hóa lại mỗi kernel để nó có *cùng độ khuếch đại* trên một biên bậc lý tưởng (tức là chia cho tổng các trọng số dương của nó) &mdash; nếu không, một kernel có hệ số thô lớn hơn sẽ trông nhiễu hơn chỉ vì tỉ lệ tổng thể của nó, chứ không phải vì hình dạng.

In [ ]:
rng = np.random.default_rng(0)
flat_noisy = np.full((150, 150), 128.0) + rng.normal(0, 15, (150, 150))

kernels = {
    'Simple [-1,0,1]': simple_x,
    'Prewitt': prewitt_x,
    'Sobel': np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=np.float64),
    'Scharr': np.array([[-3, 0, 3], [-10, 0, 10], [-3, 0, 3]], dtype=np.float64),
}

print(f'{"kernel":>18} {"noise std (gain-normalized)":>30}')
for name, k in kernels.items():
    gain = k[k > 0].sum()
    response = cv2.filter2D(flat_noisy, cv2.CV_64F, k / gain)
    print(f'{name:>18} {response.std():>30.2f}')

Kernel hiệu số đơn thuần nhiễu hơn hẳn bất kỳ toán tử $3\times3$ nào &mdash; lấy trung bình trên 3 hàng (hoặc cột) trong khi lấy hiệu làm giảm đáng kể phản hồi với nhiễu, và đó chính là mục đích của việc đưa phép làm trơn vào kernel đạo hàm.

## Từ gradient đến biên: bộ phát hiện Canny

Chỉ phân ngưỡng độ lớn gradient sẽ cho ra các mảng biên dày và nhiễu. Bộ phát hiện biên **Canny** (<a href="../references.html#canny-1986">Canny, 1986</a><span class="landmark-paper">&#9733;</span>) tinh chỉnh điều này bằng một quy trình nhiều giai đoạn:

1. **Làm trơn** bằng bộ lọc Gauss, để triệt nhiễu trước khi lấy đạo hàm.
2. **Tính gradient** (dùng Sobel bên trong) để có độ lớn và hướng tại mọi điểm ảnh.
3. **Triệt tiêu điểm không cực đại (non-maximum suppression)**: tại mỗi điểm ảnh, chỉ giữ lại độ lớn gradient nếu nó là cực đại cục bộ *dọc theo hướng gradient* &mdash; điều này làm mảnh các gờ gradient rộng xuống thành các đường chỉ rộng một điểm ảnh.
4. **Phân ngưỡng kép + trễ (hysteresis)**: các điểm ảnh trên ngưỡng cao chắc chắn là biên; các điểm ảnh dưới ngưỡng thấp bị loại bỏ; các điểm ảnh ở giữa chỉ được giữ lại nếu chúng nối liền với một biên chắc chắn. Điều này liên kết các đoạn biên yếu-nhưng-thật trong khi triệt tiêu các phản hồi nhiễu cô lập.

In [ ]:
calvin = cv2.imread('../img/calvin.png', cv2.IMREAD_GRAYSCALE)

calvin_sobel_mag = np.sqrt(cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 1, 0, ksize=3)**2 +
                           cv2.Sobel(calvin.astype(np.float64), cv2.CV_64F, 0, 1, ksize=3)**2)
naive_edges = (calvin_sobel_mag > 150).astype(np.uint8) * 255  # crude: just threshold |gradient|
canny_edges = cv2.Canny(calvin, threshold1=80, threshold2=160)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
for ax, im, title in zip(axes, [calvin, naive_edges, canny_edges],
                          ['Original', 'Naive: threshold |gradient|\n(thick, blobby)', 'Canny\n(thin, connected)']):
    ax.imshow(im, cmap='gray')
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

<p class="image-attribution">Nguồn ảnh: <a href="https://www.nga.gov/artworks/37890-john-calvin" target="_blank" rel="noopener">National Gallery of Art</a></p>

Ảnh động này minh họa quy trình Canny áp dụng trên một vùng cắt nhỏ. (Vì `cv2.Canny` không cho truy cập các giai đoạn trung gian, phần triệt tiêu điểm không cực đại và trễ được cài đặt từ đầu ở đây.) Lưu ý rằng bước trễ lan truyền thông tin tương tự như tô loang (Bài 4) -- hãy quan sát các biên mạnh (trắng) lần lượt "kết nạp" từng điểm ảnh yếu liên thông (cam), trong khi các phản hồi nhiễu yếu cô lập (tím đậm) bị bỏ lại.

<img src="../animations/canny_stages_demo.gif" alt="Ảnh động minh họa quy trình Canny 4 giai đoạn: làm trơn, độ lớn gradient, triệt tiêu điểm không cực đại, và bước trễ dạng động trong đó các biên mạnh kết nạp các điểm ảnh yếu liên thông" style="display: block; margin: 0 auto; max-width: 480px;">

### Độ nhạy với ngưỡng

Hai ngưỡng của Canny đánh đổi giữa độ đầy đủ và nhiễu. Quá thấp, nhiễu sẽ bị bắt thành các biên giả; quá cao, các biên thật (nhưng độ tương phản thấp) sẽ bị bỏ sót.

In [ ]:
noisy_calvin = np.clip(calvin.astype(np.float64) + rng.normal(0, 8, calvin.shape), 0, 255).astype(np.uint8)

threshold_pairs = [(20, 60), (80, 160), (150, 220)]

fig, axes = plt.subplots(1, len(threshold_pairs), figsize=(10, 3.5))
for ax, (lo, hi) in zip(axes, threshold_pairs):
    edges = cv2.Canny(noisy_calvin, lo, hi)
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'thresholds=({lo}, {hi})', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Với ngưỡng thấp trên ảnh nhiễu, nhiễu lấm tấm tồn tại dưới dạng các mảnh biên giả rải rác khắp nơi, đặc biệt ở phần nền phẳng. Ở các ngưỡng cao nhất, chi tiết mờ (các ô cửa sổ, các nét gạch mịn tạo bóng cho rèm và áo choàng) biến mất hoàn toàn, trong khi đường viền tối đậm của nhân vật và các cạnh của những cuốn sách vẫn tồn tại &mdash; một ví dụ thực tế về việc ngưỡng của Canny đánh đổi giữa khả năng loại bỏ nhiễu và độ nhạy với các biên mờ-nhưng-thật.

## Từ biên đến đường thẳng: biến đổi Hough

Phát hiện biên tìm ra *nơi* cường độ thay đổi gắt, nhưng nó không biết rằng một tập rải rác các điểm ảnh biên tạo thành một đường thẳng. **Biến đổi Hough** (<a href="../references.html#hough-1962">Hough, 1962</a>; <a href="../references.html#duda-hart-1972">Duda & Hart, 1972</a>) trả lời một câu hỏi khác: cho một tập điểm biên, những điểm nào phù hợp với việc cùng nằm trên một đường thẳng chung?

Hãy tham số hóa đường thẳng không phải bằng $y=mx+b$ (vốn bùng nổ với đường thẳng đứng) mà bằng khoảng cách từ gốc tọa độ và góc của pháp tuyến:

$$\rho = x\cos\theta + y\sin\theta$$

Với một điểm biên *cố định* $(x,y)$, phương trình này vẽ ra một đường cong hình sin trong không gian $(\rho,\theta)$ khi $\theta$ thay đổi &mdash; mỗi cặp $(\rho,\theta)$ trên đường cong đó mô tả một đường thẳng đi qua $(x,y)$. Ý tưởng then chốt: nếu một vài điểm biên **thẳng hàng (collinear)**, các đường sin của chúng đều cắt nhau tại *cùng một* $(\rho,\theta)$ &mdash; chính là tham số của đường thẳng chung của chúng.

In [ ]:
line_points = [(30, 270), (150, 150), (270, 30)]  # three points on the same line, x + y = 300

# the line's true (rho, theta): its normal direction is (1,1)/sqrt(2), i.e. theta=45 degrees,
# and rho is the line's distance from the origin along that normal
theta_true = np.radians(45)
rho_true = 300 * np.cos(theta_true)

thetas = np.linspace(0, np.pi, 500)
plt.figure(figsize=(6, 3.5))
for x, y in line_points:
    rho = x * np.cos(thetas) + y * np.sin(thetas)
    plt.plot(np.degrees(thetas), rho, label=f'({x},{y})')
plt.axvline(45, color='gray', linestyle='--', linewidth=1)
plt.scatter([np.degrees(theta_true)], [rho_true], color='black', zorder=5, marker='x', s=80,
            label=f'true line: rho={rho_true:.1f}, theta=45°')
plt.xlabel('theta (degrees)')
plt.ylabel('rho')
plt.legend(fontsize=8)
plt.title('Each point traces a sinusoid; collinear points cross at one (rho, theta)')
plt.show()

# confirm all three curves genuinely pass through that point, not just visually
for x, y in line_points:
    rho_at_true_theta = x * np.cos(theta_true) + y * np.sin(theta_true)
    print(f'point ({x:>3},{y:>3}): rho at theta=45 deg = {rho_at_true_theta:.2f}  (true rho = {rho_true:.2f})')

### Bộ tích lũy: bỏ phiếu cho các đường thẳng

Trên thực tế, mỗi điểm ảnh biên bỏ phiếu cho toàn bộ đường sin các đường thẳng $(\rho,\theta)$ khả dĩ của nó, vào một mảng **bộ tích lũy (accumulator)** 2D đã rời rạc hóa. Ô có nhiều phiếu nhất là đường thẳng phù hợp nhất với dữ liệu. Ta xây dựng điều này từ đầu cho một đường thẳng tổng hợp duy nhất, và đối chiếu với `cv2.HoughLines`.

In [ ]:
line_img = np.zeros((300, 300), dtype=np.uint8)
cv2.line(line_img, (30, 270), (270, 30), 255, 3)  # the same x+y=300 line as above

ys, xs = np.nonzero(line_img)
theta_bins = np.linspace(0, np.pi, 180, endpoint=False)
max_rho = int(np.hypot(*line_img.shape))
accumulator = np.zeros((2 * max_rho, len(theta_bins)), dtype=np.int32)

for x, y in zip(xs, ys):
    rho_vals = (x * np.cos(theta_bins) + y * np.sin(theta_bins)).astype(int) + max_rho
    accumulator[rho_vals, np.arange(len(theta_bins))] += 1

peak_rho_idx, peak_theta_idx = np.unravel_index(np.argmax(accumulator), accumulator.shape)
manual_rho, manual_theta = peak_rho_idx - max_rho, theta_bins[peak_theta_idx]

cv_lines = cv2.HoughLines(line_img, 1, np.pi / 180, threshold=100)
cv_rho, cv_theta = cv_lines[0, 0]

print(f'true line parameters:    rho={rho_true:.1f}, theta={np.degrees(theta_true):.1f} deg')
print(f'manual accumulator peak: rho={manual_rho}, theta={np.degrees(manual_theta):.1f} deg')
print(f'cv2.HoughLines result:   rho={cv_rho:.0f}, theta={np.degrees(cv_theta):.1f} deg')

# recover two far-apart points on the detected line, to draw it back on the image
a, b = np.cos(cv_theta), np.sin(cv_theta)
x0, y0 = a * cv_rho, b * cv_rho
pt1 = (int(x0 + 1000 * (-b)), int(y0 + 1000 * a))
pt2 = (int(x0 - 1000 * (-b)), int(y0 - 1000 * a))
overlay = cv2.cvtColor(line_img, cv2.COLOR_GRAY2BGR)
cv2.line(overlay, pt1, pt2, (0, 0, 255), 3)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(line_img, cmap='gray')
axes[0].set_title('Input: single line', fontsize=10)
axes[0].axis('off')

# most bins get only 1-2 votes from individual points' sinusoids, while the peak bin gets one
# vote per point on the line (~240 here) -- on a linear color scale that peak washes out
# everything else, so we log-compress the counts to make the fainter sinusoid trails visible too
axes[1].imshow(np.log1p(accumulator), cmap='hot', aspect='auto',
               extent=[0, 180, -max_rho, max_rho], origin='lower')
axes[1].scatter([np.degrees(theta_true)], [rho_true], edgecolor='cyan', facecolor='none', s=150,
                 linewidth=1.5, label='true (rho, theta)')
axes[1].set_xlabel('theta (degrees)')
axes[1].set_ylabel('rho')
axes[1].legend(fontsize=7)
axes[1].set_title('Accumulator (log-scaled)', fontsize=10)

axes[2].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
axes[2].set_title('Recovered line (red),\noverlaid on input', fontsize=10)
axes[2].axis('off')
plt.tight_layout()
plt.show()

Dưới đây là quá trình bỏ phiếu dưới dạng ảnh động: các điểm biên được thêm vào lần lượt từng điểm, mỗi điểm bỏ toàn bộ đường sin phiếu của mình vào bộ tích lũy, cho đến khi ô đỉnh xuất hiện tại ($\rho$, $\theta$) thực của đường thẳng.

<img src="../animations/hough_voting_demo.gif" alt="Ảnh động bộ tích lũy Hough: các điểm biên được thêm vào lần lượt, mỗi điểm bỏ một đường cong phiếu hình sin, và các phiếu hội tụ về (rho, theta) thực của đường thẳng" style="display: block; margin: 0 auto; max-width: 680px;">

Như bạn thấy, thuật toán Hough cơ bản trả về một đường thẳng vô hạn (đầu ra màu đỏ) chứ không phải một đoạn thẳng (đầu vào màu trắng).

### `cv2.HoughLinesP`: phiên bản thực dụng

Biến đổi Hough xác suất (probabilistic Hough transform) trả về thêm các điểm đầu mút của *đoạn* thẳng (không chỉ đường thẳng vô hạn), và là thứ thường được dùng trong thực tế. Ta thử nghiệm nó trên một cảnh có hai đường thẳng với hướng đã biết, sau khi chạy qua một bộ phát hiện biên Canny thực sự (biến đổi Hough hoạt động trên bản đồ biên, không phải ảnh thô).

In [ ]:
shapes_with_lines = np.zeros((300, 300), dtype=np.uint8)
cv2.line(shapes_with_lines, (30, 270), (270, 30), 200, 2)   # true angle: -45 degrees
cv2.line(shapes_with_lines, (50, 50), (250, 50), 200, 2)    # true angle: 0 degrees
cv2.circle(shapes_with_lines, (220, 220), 30, 150, -1)      # a distractor shape, no straight edges

edges = cv2.Canny(shapes_with_lines, 80, 160)
segments = cv2.HoughLinesP(edges, 1, np.pi / 180, threshold=60, minLineLength=80, maxLineGap=5)

# color each segment by which of the two true lines (-45 or 0 degrees) it's closer to
# (colors given in BGR, since `vis` gets converted to RGB before display below)
color_diagonal, color_horizontal = (255, 0, 0), (80, 80, 255)  # red, blue

vis = cv2.cvtColor(shapes_with_lines, cv2.COLOR_GRAY2BGR)
print(f'{"segment":>20} {"angle (deg)":>12}')
for x1, y1, x2, y2 in segments.reshape(-1, 4):
    angle = np.degrees(np.arctan2(y2 - y1, x2 - x1))
    print(f'({x1},{y1})-({x2},{y2})'.rjust(20), f'{angle:>12.1f}')
    color = color_diagonal if abs(angle - (-45)) < abs(angle - 0) else color_horizontal
    cv2.line(vis, (x1, y1), (x2, y2), color, 2)

fig, axes = plt.subplots(1, 3, figsize=(9, 3.5))
for ax, im, title in zip(axes, [shapes_with_lines, edges, vis],
                          ['Original', 'Canny edges', 'Detected segments\n(red = -45°, blue = 0°)']):
    ax.imshow(im, cmap='gray' if im.ndim == 2 else None)
    ax.set_title(title, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

Mọi đoạn thẳng phát hiện được đều rơi gần như chính xác vào một trong hai góc thực của đường thẳng (-45 và 0 độ) &mdash; mỗi đường thẳng thường cho ra hai hoặc ba phát hiện gần trùng nhau thay vì một, vì một đường vẽ có độ rộng hữu hạn, nên Canny tìm thấy một biên ở *mỗi* bên của nét vẽ, cách nhau vài điểm ảnh. Các quy trình thực tế dựa trên Hough thường gộp các phát hiện gần nhau trong một bước hậu xử lý chính vì lý do này. Hình tròn &mdash; vốn không có cạnh thẳng nào &mdash; đúng là không tạo ra phát hiện đường thẳng nào cả.

### Vượt ra ngoài đường thẳng: Hough cho đường tròn

Cùng ý tưởng bỏ phiếu đó tổng quát hóa được cho bất kỳ hình dạng nào mô tả được bằng một số ít tham số. Một đường tròn cần 3 tham số (tọa độ tâm $x$, tọa độ tâm $y$, bán kính $r$), nên `cv2.HoughCircles` tích lũy phiếu trong không gian tham số 3D thay vì 2D.

In [ ]:
circle_img = np.zeros((200, 200), dtype=np.uint8)
true_center, true_radius = (100, 100), 50
cv2.circle(circle_img, true_center, true_radius, 255, 1)

detected = cv2.HoughCircles(circle_img, cv2.HOUGH_GRADIENT, dp=1, minDist=50,
                             param1=50, param2=20, minRadius=30, maxRadius=70)
cx, cy, r = detected[0, 0]

print(f'true circle:      center={true_center}, radius={true_radius}')
print(f'detected circle:  center=({cx:.1f}, {cy:.1f}), radius={r:.1f}')

vis = cv2.cvtColor(circle_img, cv2.COLOR_GRAY2BGR)
cv2.circle(vis, (int(cx), int(cy)), int(r), (0, 0, 255), 1)
cv2.drawMarker(vis, (int(cx), int(cy)), (0, 0, 255), cv2.MARKER_CROSS, 10)
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title('Detected circle (red) overlaid on the true one')
plt.axis('off')
plt.show()

### Nhìn vào bên trong: bộ tích lũy đường tròn

`cv2.HoughCircles` không cho truy cập bộ tích lũy bên trong, nhưng tự xây dựng một bộ từ đầu cũng là ý tưởng bỏ phiếu như bộ tích lũy đường thẳng ở trên, chỉ thêm một chiều: đường tròn cần một tâm *và* một bán kính, nên mỗi điểm ảnh biên bỏ phiếu cho mọi bộ ba $(c_x, c_y, r)$ mà nó có thể phù hợp, với mọi bán kính ứng viên $r$.

In [ ]:
circle_edges = cv2.Canny(circle_img, 50, 100)
edge_ys, edge_xs = np.nonzero(circle_edges)

H, W = circle_img.shape
radii = np.arange(30, 71)
vote_thetas = np.linspace(0, 2 * np.pi, 72, endpoint=False)  # angle around each candidate circle
cos_t, sin_t = np.cos(vote_thetas), np.sin(vote_thetas)

circle_accumulator = np.zeros((len(radii), H, W), dtype=np.int32)
for ri, r in enumerate(radii):
    # for this radius, every edge point votes for every center that's r away from it
    cand_cx = np.round(edge_xs[:, None] - r * cos_t[None, :]).astype(int)
    cand_cy = np.round(edge_ys[:, None] - r * sin_t[None, :]).astype(int)
    valid = (cand_cx >= 0) & (cand_cx < W) & (cand_cy >= 0) & (cand_cy < H)
    np.add.at(circle_accumulator[ri], (cand_cy[valid], cand_cx[valid]), 1)

peak_r_idx, peak_cy, peak_cx = np.unravel_index(np.argmax(circle_accumulator), circle_accumulator.shape)
print(f'true circle:       center={true_center}, radius={true_radius}')
print(f'accumulator peak:  center=({peak_cx}, {peak_cy}), radius={radii[peak_r_idx]}')

radius_vs_cx = circle_accumulator.max(axis=1)  # max over cy
radius_vs_cy = circle_accumulator.max(axis=2)  # max over cx

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(radius_vs_cx, aspect='auto', cmap='hot', extent=[0, W, radii[-1], radii[0]])
axes[0].set_xlabel('cx'); axes[0].set_ylabel('radius')
axes[0].set_title('radius vs. cx (max over cy)', fontsize=10)
axes[1].imshow(radius_vs_cy, aspect='auto', cmap='hot', extent=[0, H, radii[-1], radii[0]])
axes[1].set_xlabel('cy'); axes[1].set_ylabel('radius')
axes[1].set_title('radius vs. cy (max over cx)', fontsize=10)
plt.tight_layout()
plt.show()

Để tránh khó khăn khi trực quan hóa một mảng 3D, ta lấy các lát cắt 2D: số phiếu cực đại theo $c_y$ (cho ra bán kính theo $c_x$) và theo $c_x$ (cho ra bán kính theo $c_y$). Mỗi lát cắt cho thấy một chữ X sáng: phiếu của mỗi điểm biên vẽ ra hai đường chéo khi bán kính ứng viên tăng lên (một cho các tâm ở bên trái nó, một cho các tâm ở bên phải), và tất cả cắt nhau tại tâm và bán kính thực, chính là điểm sáng nhất trong mỗi đồ thị.

### Bài tập

1. Tăng độ lệch chuẩn của nhiễu trong `flat_noisy` và chạy lại phép so sánh khả năng chống nhiễu. Thứ tự tương đối của bốn kernel có thay đổi không?
2. `cv2.Canny` nhận tham số `apertureSize` cho bước Sobel bên trong (mặc định là 3). Hãy thử `apertureSize=5` và so sánh kết quả với mặc định trên ảnh nhiễu.
3. Bước trễ của Canny cần một đường đi *liên thông* gồm các điểm ảnh trên ngưỡng thấp giữa một biên yếu và một biên mạnh. Hãy dựng một ví dụ nhị phân nhỏ (bằng tay, dưới dạng mảng NumPy) trong đó một biên thật bị đứt thành hai đoạn với khoảng hở 1 điểm ảnh, và xác nhận rằng bước trễ không nối được chúng dù người xem thấy rõ đó là một biên.
4. Giảm tham số `threshold` của `cv2.HoughLinesP` xuống thấp hơn nhiều so với 60 trên `shapes_with_lines`. Bạn có bắt đầu nhận được các đoạn ngắn giả do nhiễu trên đường biên Canny của hình tròn không, dù nó không có cạnh thẳng nào? Tăng `minLineLength` giúp khắc phục điều này như thế nào?
5. Thêm một đường thẳng thứ ba vào `shapes_with_lines` với góc nghiêng thoải (ví dụ từ `(30, 200)` tới `(270, 150)`), và dự đoán góc của nó trước khi đối chiếu đầu ra của `HoughLinesP` với dự đoán của bạn.
6. Tham số `param2` của `cv2.HoughCircles` gần giống một ngưỡng phiếu của bộ tích lũy (như `threshold` của `HoughLines`) &mdash; giá trị thấp hơn sẽ báo cáo nhiều đường tròn hơn (và nhiều đường tròn giả hơn). Hãy quét `param2` từ 30 xuống 10 trên một phiên bản nhiễu của `circle_img` (thêm nhiễu Gauss trước) và quan sát khi nào các đường tròn dương tính giả bắt đầu xuất hiện.